In [ ]:
# Candidate Elimination Algorithm

def more_general_or_equal(h1, h2):
    """Returns True if h1 is more general than or equal to h2."""
    return all(x == '?' or x == y for x, y in zip(h1, h2))


def covers(hypothesis, example):
    """Check whether a hypothesis covers an example."""
    return all(h == '?' or h == x for h, x in zip(hypothesis, example))


def minimal_generalizations(s, example):
    """Generate minimal generalizations of S to cover a positive example."""
    new_s = list(s)

    for i in range(len(s)):
        if s[i] == '∅':
            new_s[i] = example[i]
        elif s[i] != example[i]:
            new_s[i] = '?'

    return tuple(new_s)


def minimal_specializations(g, example, domains):
    """Generate minimal specializations of G to exclude a negative example."""
    specializations = []

    for i in range(len(g)):
        if g[i] == '?':
            for value in domains[i]:
                if value != example[i]:
                    new_g = list(g)
                    new_g[i] = value
                    specializations.append(tuple(new_g))

    return specializations


def candidate_elimination(data, domains):
    num_attributes = len(data[0][0])

    # Most specific and most general hypotheses
    S = {tuple('∅' for _ in range(num_attributes))}
    G = {tuple('?' for _ in range(num_attributes))}

    for example, label in data:

        if label == 'Yes':
            # Remove hypotheses from G that don't cover the positive example
            G = {g for g in G if covers(g, example)}

            new_S = set()

            for s in S:
                if covers(s, example):
                    new_S.add(s)
                else:
                    generalized = minimal_generalizations(s, example)

                    # Keep only hypotheses that are more specific than
                    # at least one hypothesis in G
                    for h in [generalized]:
                        if any(more_general_or_equal(g, h) for g in G):
                            new_S.add(h)

            S = new_S

        else:  # Negative example
            # Remove hypotheses from S that cover the negative example
            S = {s for s in S if not covers(s, example)}

            new_G = set()

            for g in G:
                if not covers(g, example):
                    new_G.add(g)
                else:
                    specializations = minimal_specializations(
                        g, example, domains
                    )

                    for h in specializations:
                        if any(more_general_or_equal(h, s) for s in S):
                            new_G.add(h)

            G = new_G

    return S, G


# -------------------------
# Example Dataset
# -------------------------

data = [
    (('Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same'), 'Yes'),
    (('Sunny', 'Warm', 'High', 'Strong', 'Warm', 'Same'), 'Yes'),
    (('Rainy', 'Cold', 'High', 'Strong', 'Warm', 'Change'), 'No'),
    (('Sunny', 'Warm', 'High', 'Weak', 'Cool', 'Change'), 'Yes')
]

domains = [
    ['Sunny', 'Rainy'],       # Sky
    ['Warm', 'Cold'],         # AirTemp
    ['Normal', 'High'],       # Humidity
    ['Strong', 'Weak'],       # Wind
    ['Warm', 'Cold'],         # Water
    ['Same', 'Change']        # Forecast
]

S, G = candidate_elimination(data, domains)

print("Specific Boundary (S):")
for hypothesis in S:
    print(hypothesis)

print("\nGeneral Boundary (G):")
for hypothesis in G:
    print(hypothesis)


Specific Boundary (S):
('Sunny', 'Warm', '?', '?', '?', '?')

General Boundary (G):
('?', 'Warm', '?', '?', '?', '?')
('Sunny', '?', '?', '?', '?', '?')
